# Laboratory 07 — The second law and heat engines

In this laboratory you will build heat engines, measure their efficiencies, and try — and
fail — to design one that beats the Carnot bound.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. A prediction you have committed to is the
only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | a fixed amount of ideal gas, $N$ particles with $f$ quadratic degrees of freedom |
| **Dynamics** | quasistatic strokes joined into a closed loop in the $P$–$V$ plane |
| **Boundary** | frictionless piston; wall switched between diathermal and adiabatic |
| **Ensemble** | not applicable — this is thermodynamics, nothing counts microstates |
| **Ignored** | friction, piston mass, gas non-ideality, leaks through the adiabats, the time a stroke takes |
| **Valid when** | strokes are slow compared with the relaxation time; reservoirs are large enough not to change temperature |
| **Failure modes** | finite-rate operation, regenerators, a working substance near condensation |

All the physics lives in `thermolab.engines` — open it and read it. Nothing in this course is
hidden inside a framework.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import engines
from thermolab.validation import relative_error

T_HOT = 600.0   # K — the hot reservoir
T_COLD = 300.0  # K — the cold reservoir
N_PARTICLES = 1000
V_START = 1.0e-3  # m^3

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"Carnot bound between {T_HOT:.0f} K and {T_COLD:.0f} K: "
      f"{engines.carnot_efficiency(T_HOT, T_COLD):.6f}")

## Part 1 — Build a Carnot engine and look at it

Four strokes: expand in contact with the hot reservoir, expand with the heat shut off,
compress against the cold reservoir, compress with the heat shut off again. The loop closes,
and the area it encloses is the work delivered.

In [ ]:
cycle = engines.carnot_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, expansion_ratio=2.5)

fig, (plane, bars) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.4, 1]})
for stroke, colour in zip(cycle.strokes,
                          ["#dc2626", "#94a3b8", "#2563eb", "#94a3b8"], strict=True):
    path = stroke.process.quasistatic_path
    plane.plot(path.volumes * 1e3, path.pressures, lw=2.4, color=colour)
plane.set_xlabel("volume (L)")
plane.set_ylabel("pressure (Pa)")
plane.set_title("the cycle")

bars.bar(["heat in", "work out", "heat dumped"],
         [cycle.heat_absorbed, cycle.work_output, cycle.heat_rejected],
         color=["#dc2626", "#0f172a", "#2563eb"])
bars.set_ylabel("energy per cycle (J)")
bars.set_title("the ledger")
plt.tight_layout()
plt.show()

print(f"heat absorbed   Q_h = {cycle.heat_absorbed:.4e} J")
print(f"work delivered  W   = {cycle.work_output:.4e} J")
print(f"heat rejected   Q_c = {cycle.heat_rejected:.4e} J")
print(f"efficiency          = {cycle.efficiency:.6f}")
print(f"Carnot bound        = {cycle.carnot_bound:.6f}")

Notice that the work bar is exactly half the heat-in bar, and that the heat-dumped bar is the
other half. The engine is *perfect* — every stroke is reversible — and it still throws away
half of what it took in.

### Predict

Before running the next cell: you are about to build the same engine out of a different gas
(monatomic, diatomic, polyatomic) and with different expansion ratios. Which of these changes
the efficiency, and in which direction?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
print(f"{'f':>3} {'ratio':>7} {'N':>8}   efficiency")
print("-" * 40)
for dof in (3, 5, 6):
    for ratio in (1.2, 2.5, 8.0):
        eta = engines.carnot_cycle(
            N_PARTICLES, T_HOT, T_COLD, V_START, ratio, degrees_of_freedom=dof
        ).efficiency
        print(f"{dof:>3} {ratio:>7.1f} {N_PARTICLES:>8}   {eta:.12f}")

# And over four decades of engine size, at fixed gas and shape:
for n in (10, 1000, 100_000):
    eta = engines.carnot_cycle(n, T_HOT, T_COLD, V_START, 2.5).efficiency
    print(f"{3:>3} {2.5:>7.1f} {n:>8}   {eta:.12f}")

Twelve figures, every time. The gas does not matter; the size does not matter; the shape of
the loop does not matter. This is Carnot's theorem, and it is worth sitting with: the proof on
the module page never opens the engine, so nothing about the engine's insides can appear in
the answer.

The heats themselves are *not* the same — a bigger engine moves proportionally more energy.
It is their ratio that is fixed.

## Part 2 — Try to beat the bound

Now the interesting experiment. Build engines at random: any size, any expansion ratio, any
quality of thermal contact. See if any of them exceeds $1 - T_c/T_h$.

In [ ]:
efficiencies = []
for seed in range(200):
    engine = engines.random_two_reservoir_engine(np.random.default_rng(seed), T_HOT, T_COLD)
    efficiencies.append(engine.efficiency)

efficiencies = np.array(efficiencies)
bound = engines.carnot_efficiency(T_HOT, T_COLD)

plt.figure(figsize=(7, 4))
plt.hist(efficiencies, bins=30, color="#2563eb", alpha=0.75)
plt.axvline(bound, color="#dc2626", lw=2.2, label=f"Carnot bound = {bound:.3f}")
plt.xlabel("measured efficiency")
plt.ylabel("engines")
plt.legend()
plt.show()

print(f"engines built:            {efficiencies.size}")
print(f"best efficiency found:    {efficiencies.max():.6f}")
print(f"Carnot bound:             {bound:.6f}")
print(f"how many exceeded it:     {(efficiencies > bound).sum()}")

None of them. Try changing the seed range, the reservoir temperatures, the gas — the wall does
not move.

Be careful about what this shows. Two hundred engines respecting a bound is **not** a proof
that it cannot be beaten; no finite sample could be, and every engine here was built by code
that already implements the physics correctly. What the sweep really is, is a *falsification
test*: if you could make one exceed the bound, either the library or the derivation would be
wrong. Failing to break something you tried hard to break is evidence, not proof.

## Part 3 — Run it backwards

Every stroke of a reversible cycle can be reversed. Do that and the engine becomes a
refrigerator: work goes in, heat comes out of the cold side.

In [ ]:
fridge = engines.reversed_carnot_cycle(N_PARTICLES, 298.0, 275.0, V_START, 2.5)

print(f"work consumed per cycle   = {-fridge.work_output:.4e} J")
print(f"heat lifted from the cold = {fridge.heat_absorbed:.4e} J")
print(f"coefficient of performance = {fridge.coefficient_of_performance:.4f}")
print(f"closed form T_c/(T_h-T_c)  = {engines.cop_refrigerator(298.0, 275.0):.4f}")
print()
print(f"as a heat pump, COP        = {engines.cop_heat_pump(298.0, 275.0):.4f}")
print(f"difference is exactly 1:     "
      f"{engines.cop_heat_pump(298.0, 275.0) - engines.cop_refrigerator(298.0, 275.0):.10f}")

# Asking a refrigerator for an efficiency is a category error, and the library says so.
try:
    _ = fridge.efficiency  # bound only so the access is not a bare expression
except ValueError as error:
    print(f"\nasking for its efficiency: {error}")

### Does the fridge break the second law?

It genuinely lowers the entropy of the food inside it. Run the bookkeeping and see where the
compensation comes from — this is the experiment that kills the "a fridge violates the second
law" reading, because both entropies are computed, not asserted.

In [ ]:
T_COLD_IN, T_KITCHEN = 275.0, 298.0
heat_lifted = 1000.0  # J removed from the food

for label, cop in [("a real fridge", 3.2), ("the best possible fridge",
                                            engines.cop_refrigerator(T_KITCHEN, T_COLD_IN))]:
    work = heat_lifted / cop
    dumped = heat_lifted + work           # first law: everything lifted, plus the work
    food = -heat_lifted / T_COLD_IN       # the food's entropy really does fall
    kitchen = dumped / T_KITCHEN          # the kitchen's rises
    print(f"{label} (COP {cop:.2f}):")
    print(f"    food    dS = {food:+.4f} J/K")
    print(f"    kitchen dS = {kitchen:+.4f} J/K")
    total = food + kitchen
    # The reversible fridge sits exactly at zero, so what comes back there is rounding of
    # either sign. A bare "-0.0000" would read as precisely the violation this cell rules
    # out, so display it as the zero it is.
    shown = 0.0 if abs(total) < 1e-9 * abs(food) else total
    print(f"    total   dS = {shown:+.4f} J/K")
    assert food < 0.0                     # the misconception's premise is TRUE
    assert total > -1e-9 * abs(food)      # ...and its conclusion still does not follow
    print()

print("The premise holds and the conclusion fails: the second law constrains the total.")
print("Only the reversible fridge reaches zero, and nothing gets below it.")

A coefficient of performance of about 12 is not a violation of anything. Nothing is being
*converted* here — energy is being **moved**, and moving heat uphill costs less than the
amount moved. That is also why a heat pump heats a house for a fraction of what a resistive
heater costs.

## Part 4 — Where the lost work goes

A real engine cannot touch its reservoirs at exactly their temperatures: heat will not cross a
zero temperature difference at a finite rate. Give the gas a gap at each end and watch two
things move together.

In [ ]:
gaps = np.linspace(0.0, 100.0, 26)
etas, produced = [], []
for gap in gaps:
    engine = engines.endoreversible_cycle(
        N_PARTICLES, T_HOT, T_COLD, V_START, 2.5, hot_gap=float(gap), cold_gap=float(gap)
    )
    etas.append(engine.efficiency)
    produced.append(engine.entropy_produced)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(gaps, etas, lw=2.2, color="#2563eb")
left.axhline(engines.carnot_efficiency(T_HOT, T_COLD), ls="--", color="#dc2626",
             label="Carnot bound")
left.set_xlabel("temperature gap at each end (K)")
left.set_ylabel("efficiency")
left.legend()

right.plot(gaps, produced, lw=2.2, color="#d97706")
right.set_xlabel("temperature gap at each end (K)")
right.set_ylabel("entropy produced per cycle (J/K)")
plt.tight_layout()
plt.show()

The efficiency falls and the entropy produced rises, together. They are not two effects but
one, and the exact relation is worth checking directly — lost work equals the cold reservoir's
temperature times the entropy produced (the Gouy–Stodola result).

In [ ]:
engine = engines.endoreversible_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, 2.5, 40.0, 40.0)
perfect = engines.carnot_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, 2.5)

# Scale both to the same heat absorbed so the comparison is fair.
scaled_perfect_work = perfect.efficiency * engine.heat_absorbed
lost = scaled_perfect_work - engine.work_output

print(f"work delivered by the real engine   = {engine.work_output:.6e} J")
print(f"a perfect engine on the same heat   = {scaled_perfect_work:.6e} J")
print(f"work lost                           = {lost:.6e} J")
print(f"T_c * entropy produced              = {T_COLD * engine.entropy_produced:.6e} J")
print(f"relative difference                 = "
      f"{relative_error(lost, T_COLD * engine.entropy_produced):.2e}")

# --- the checks that also live in the project's test suite ---
scale = abs(cycle.strokes[0].process.start.internal_energy)

# 1. The loop closes: four independent closed forms must agree.
assert abs(cycle.internal_energy_drift) / scale < 1e-12

# 2. The first law closes around the loop.
assert abs(cycle.first_law_residual) / scale < 1e-12

# 3. A reversible cycle produces no entropy.
assert abs(perfect.entropy_produced) / perfect.entropy_scale < 1e-12

# 4. An irreversible one produces a strictly positive amount.
assert engine.entropy_produced / engine.entropy_scale > 1e-6

# 5. The area enclosed really is the work delivered (quadrature vs closed form).
assert relative_error(perfect.enclosed_area, perfect.work_output) < 1e-4

print("\nall five checks passed")

## Part 5 — Explore it yourself

The sliders let you vary the reservoirs and the quality of the thermal contact. Two
experiments worth doing:

1. Bring the two reservoir temperatures close together and watch the efficiency collapse.
   This is why low-grade waste heat is nearly worthless however much of it there is.
2. Hold the temperatures fixed and open up the gaps. Watch how much efficiency you lose for
   the privilege of running at a finite rate.

Press **Run Interact** after moving the sliders — the callback redraws two panels.

In [ ]:
import ipywidgets as widgets


def explore(t_hot=600.0, t_cold=300.0, gap=20.0, expansion_ratio=2.5):
    span = t_hot - t_cold
    gap = min(gap, 0.45 * span)  # keep the working temperatures from crossing
    engine = engines.endoreversible_cycle(
        N_PARTICLES, t_hot, t_cold, V_START, expansion_ratio,
        hot_gap=gap, cold_gap=gap,
    )
    bound = engines.carnot_efficiency(t_hot, t_cold)

    fig, (plane, bars) = plt.subplots(1, 2, figsize=(11, 3.8),
                                      gridspec_kw={"width_ratios": [1.4, 1]})
    for stroke, colour in zip(engine.strokes,
                              ["#dc2626", "#94a3b8", "#2563eb", "#94a3b8"],
                              strict=True):
        path = stroke.process.quasistatic_path
        plane.plot(path.volumes * 1e3, path.pressures, lw=2.2, color=colour)
    plane.set_xlabel("volume (L)")
    plane.set_ylabel("pressure (Pa)")

    bars.bar(["this engine", "Carnot bound"], [engine.efficiency, bound],
             color=["#2563eb", "#dc2626"])
    bars.set_ylim(0, 1)
    bars.set_ylabel("efficiency")
    plt.tight_layout()
    plt.show()

    print(f"efficiency        {engine.efficiency:.4f}")
    print(f"Carnot bound      {bound:.4f}")
    print(f"entropy produced  {engine.entropy_produced:.3e} J/K per cycle")


widgets.interact_manual(
    explore,
    t_hot=widgets.FloatSlider(min=350, max=1200, step=25, value=600),
    t_cold=widgets.FloatSlider(min=200, max=340, step=5, value=300),
    gap=widgets.FloatSlider(min=0, max=100, step=5, value=20),
    expansion_ratio=widgets.FloatSlider(min=1.2, max=8.0, step=0.2, value=2.5),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "07-second-law.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. You tried to beat the Carnot bound and failed. Explain why that failure is not a proof of
   Carnot's theorem, and say what would count as one.
3. A colleague says the second law is "really just about friction and losses". Give them the
   one-sentence correction that a perfect, frictionless engine would still respect.

**Your answers:**

1.
2.
3.